# CSA Speed Benchmark

Controls timing benchmarks for `CSADataHandler` and `JourneyPlanner`.

**Workflow**
1. Run **Setup** once per kernel session.
2. Run **Prepare** only when the planner has not been prepared yet, or when you intentionally want to rebuild/refetch data.
3. After editing `src/routing/journey_planner_v3.py`, run the **reload code only** cell. It keeps the prepared in-memory data and swaps in the latest planner code.
4. Run any benchmark cell - set `iterations` to reduce noise from fluctuations.

The instrumented code inside the source files prints a per-function breakdown automatically.
The bench functions here add aggregate stats (avg / min / max / stdev).

## Setup

In [1]:
import importlib
import os
import sys

cwd = os.path.abspath(os.getcwd())
project_root = cwd if os.path.exists(os.path.join(cwd, "src")) else os.path.abspath(os.path.join(cwd, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import src.config.settings as _s
import src.data.csa_data_handler as _cdh
import src.routing.journey_planner as _jp
import tests.test_CSA as _bench


def _prepared_summary(p):
    if not getattr(p, "prepared", False):
        return " (planner is not prepared yet)"

    days = getattr(p, "connections_by_day", None) or {}
    n_day_connections = sum(len(v) for v in days.values())
    n_stops = len(getattr(p, "stops", None) or [])
    n_trips = getattr(p, "n_trips", 0)
    return f" ({n_stops:,} stops, {n_trips:,} trips, {n_day_connections:,} day-connections)"


def reload_planner_code(preserve_prepared=True):
    """Reload planner/benchmark code while keeping prepared CSA data in memory."""
    global settings, get_settings, JourneyPlanner
    global bench_prepare, bench_plan_candidates, bench_plan_profile, bench_route, bench_route_backward
    global _s, _cdh, _jp, _bench, planner

    old_planner = globals().get("planner")

    for module in (_s, _cdh, _jp, _bench):
        importlib.reload(module)

    get_settings = _s.get_settings
    JourneyPlanner = _jp.JourneyPlanner
    bench_prepare = _bench.bench_prepare
    bench_plan_candidates = _bench.bench_plan_candidates
    bench_route = _bench.bench_route
    bench_plan_profile = _bench.bench_plan_profile
    bench_route_backward = _bench.bench_route_backward
    try:
        settings = get_settings()
    except Exception:
        if old_planner is not None and hasattr(old_planner, "settings"):
            settings = old_planner.settings
        elif "settings" in globals():
            settings = globals()["settings"]
        else:
            raise

    new_planner = JourneyPlanner(settings=settings)

    if preserve_prepared and old_planner is not None and getattr(old_planner, "prepared", False):
        for name, value in vars(old_planner).items():
            setattr(new_planner, name, value)

        # Keep current settings/schema fresh, while preserving the materialized indexes.
        new_planner.settings = settings
        new_planner.schema = getattr(old_planner, "schema", new_planner.schema)
        new_planner.shared_schema = getattr(old_planner, "shared_schema", new_planner.shared_schema)

        if getattr(new_planner, "data_handler", None) is not None:
            try:
                new_planner.data_handler.__class__ = _cdh.CSADataHandler
            except TypeError:
                pass

        planner = new_planner
        print("Reloaded journey_planner_v3.py; preserved prepared data" + _prepared_summary(planner))
    else:
        planner = new_planner
        print("Reloaded journey_planner_v3.py; created a fresh unprepared planner")

    return planner


planner = reload_planner_code(preserve_prepared=True)

Reloaded journey_planner_v3.py; created a fresh unprepared planner


In [10]:
# Run this after editing src/routing/journey_planner_v3.py.
# This swaps in the latest method code while keeping prepared data in memory.
planner = reload_planner_code()

Reloaded journey_planner_v3.py; preserved prepared data (25,752 stops, 1,340,906 trips, 35,752,786 day-connections)


In [3]:
import src.config.settings

LAUSANNE_REGION_UUIDS = (
    "a7a21b73-6ffe-4fbf-a635-6e2b961f3072",
    "e168fd57-f57a-4075-a350-0dcfbb55147f",
)
REGION_UUIDS = settings.region_uuids or LAUSANNE_REGION_UUIDS

START_STOP = 8501120   # Lausanne
END_STOP   = 8501117   # Renens VD


print(f"Regions : {REGION_UUIDS}")
print(f"Stops   : {START_STOP} → {END_STOP}")

Regions : ('a7a21b73-6ffe-4fbf-a635-6e2b961f3072', 'e168fd57-f57a-4075-a350-0dcfbb55147f')
Stops   : 8501120 → 8501117


## Prepare

Run this only when `planner.prepared` is `False`, or when you intentionally want to rebuild/refetch data.

- `REBUILD_TABLES = True` -> drop and recreate Trino tables (slow)
- `REBUILD_TABLES = False` -> skip table creation, only fetch + materialize
- After editing `journey_planner_v3.py`, run the reload-code cell above instead of this cell.

In [4]:
FORCE_PREPARE = False
REBUILD_TABLES = False
REBUILD_PREREQUISITES = False

if planner.prepared and not FORCE_PREPARE:
    print("planner already prepared; skipping prepare(). Set FORCE_PREPARE=True to run it again.")
    prepare_result = {"skipped": True}
else:
    planner = reload_planner_code(preserve_prepared=False)
    prepare_result = bench_prepare(
        planner,
        regions=None,
        rebuild=REBUILD_TABLES,
        rebuild_prerequisites=REBUILD_PREREQUISITES,
    )

prepare_result

"""
On Lausanne:
    v1 40 sec
    v2 40 sec -> cache everything

On whole switzerland
   ~8min
"""

Reloaded journey_planner_v3.py; created a fresh unprepared planner

  bench_prepare


/home/kuci/project/final/src/data/csa_data_handler.py:401: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, self.conn)



  BUILD CSA TABLES
  skipping  stop_times_seq (already exists)
  build_stop_times_seq            0.10s
  skipping  stop_times_trips_seq (already exists)
  build_stop_times_trips_seq      0.08s
  skipping  full_table_seq (already exists)
  build_full_table_seq            0.08s
  skipping  connections (already exists)
  build_connections               0.08s
--------------------------------------------
  total build_all                 0.34s

  FETCH DATA
  fetch_stops                     0.59s  (25752 rows)
  fetch_footpaths                 2.01s  (52104 rows)
  fetch_connections             251.05s  (14508919 rows)

  MATERIALIZE IN-MEMORY
  stops dict                      0.04s  (25752 stops)
  footpaths dict                  0.14s  (52104 edges)
  connections loop               80.98s  (14508919 rows)
  sort + columns                 77.30s  (1340906 trips)
  trip meta by idx                0.91s
--------------------------------------------
  TOTAL prepare()               414.75s

--

'\nOn Lausanne:\n    v1 40 sec\n    v2 40 sec -> cache everything\n\nOn whole switzerland\n   ~8min\n'

## Benchmark — plan_candidates

> Note: check bellow for a more comprehensive benchmarking

## Benchmark random 200

In [8]:
import random
import statistics

random.seed(42)
stop_list = list(planner.stops)

# bias toward stops that actually have many connections (busy stops)
"""
This ensures you're benchmarking realistic journeys between actual hubs — 
Zürich, Bern, Geneva, Basel, Lausanne — which is what matters in production 
and where the optimizations actually show their effect.
"""

from collections import Counter
stop_freq = Counter()
for conn in planner.connections_by_day["monday"]:
    stop_freq[conn[0]] += 1
    stop_freq[conn[1]] += 1

# take top 200 busiest stops only
busy_stops = [s for s, _ in stop_freq.most_common(200)]

TEST_PAIRS = []
random.seed(42)
#while len(TEST_PAIRS) < 100:
while len(TEST_PAIRS) < 30:
    s, e = random.choice(busy_stops), random.choice(busy_stops)
    if s != e:
        TEST_PAIRS.append((s, e))

In [11]:
all_times = []

for i, (START_STOP, END_STOP) in enumerate(TEST_PAIRS):
    print(f"ON run: {i}")
    result = bench_plan_candidates(
        planner,
        iterations=4,
        start_stop_id=START_STOP,
        end_stop_id=END_STOP,
        travel_date="2026-05-11",
        arrival_deadline="09:00",
        max_routes=5
    )
    all_times.append(result["avg_ms"])

all_times.sort()
print(f"  n:    {len(all_times)}")
print(f"  avg:  {statistics.mean(all_times):.1f}ms")
print(f"  p50:  {all_times[len(all_times)//2]:.1f}ms")
print(f"  p95:  {all_times[int(len(all_times)*0.95)]:.1f}ms")
print(f"  max:  {all_times[-1]:.1f}ms")

ON run: 0

  bench_plan_candidates  (n=4)
  run  1/4    plan_candidates     total=243.6ms  scanned=209376  routes=5
  run  2/4    run  3/4    run  4/4  --------------------------------------------
  avg                        95.54 ms
  min                         0.02 ms
  max                       382.09 ms
  stdev                     191.04 ms
--------------------------------------------
  bench_plan_candidates done     95.54 ms

ON run: 1

  bench_plan_candidates  (n=4)
  run  1/4    plan_candidates     total=1541.3ms  scanned=950031  routes=0
  run  2/4    run  3/4    run  4/4  --------------------------------------------
  avg                       605.72 ms
  min                         0.01 ms
  max                      2422.86 ms
  stdev                    1211.42 ms
--------------------------------------------
  bench_plan_candidates done    605.72 ms

ON run: 2

  bench_plan_candidates  (n=4)
  run  1/4    plan_candidates     total=1888.4ms  scanned=1072110  routes=0
  run  

## Results




** Claude**
  - n:    30
  - avg:  589.4ms
  - p50:  331.5ms
  - p95:  2390.3ms
  - max:  2519.5ms


** Old**

  - n:    30
  - avg:  2010.2ms
  - p50:  2668.4ms
  - p95:  2814.9ms
  - max:  2820.5ms


** Claude with backward scan**

  - n:    30
  - avg:  597.0ms
  - p50:  450.5ms
  - p95:  2384.3ms
  - max:  2456.3ms


** Further optimization**
  - n:    30
  - avg:  491.0ms
  - p50:  312.4ms
  - p95:  2544.4ms
  - max:  2544.9ms

** On cluster **
 - n:    30
 - avg:  291.2ms
 - p50:  319.5ms
 - p95:  615.5ms
 - max:  626.6ms

Delays predicted here.
 - n:    30
 - avg:  549.7ms
 - p50:  359.4ms
 - p95:  2572.2ms
 - max:  2733.4ms

Full confidence val 
  - n:    30
  - avg:  559.0ms
  - p50:  391.3ms
  - p95:  2423.3ms
  - max:  2557.5ms